In [ ]:
%run "/Users/manojrammopati/Public/Projects/bupa_insurance_project/_01_Bronze/Jupyter Notebooks/00_bronze_data_connector.ipynb"

25/12/06 20:45:37 WARN Utils: Your hostname, Manojrams-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.0.219 instead (on interface en0)
25/12/06 20:45:37 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/manojrammopati/.ivy2/cache
The jars for the packages stored in: /Users/manojrammopati/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-azure added as a dependency
com.azure#azure-storage-blob added as a dependency
com.azure#azure-identity added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a615b637-3b0f-4a3f-8aa0-1d03d6d6aab5;1.0
	confs: [default]


:: loading settings :: url = jar:file:/opt/anaconda3/envs/spark_local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-azure;3.3.4 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found commons-logging#commons-logging;1.1.3 in central
	found commons-codec#commons-codec;1.15 in central
	found com.microsoft.azure#azure-storage;7.0.1 in central
	found com.fasterxml.jackson.core#jackson-core;2.12.7 in central
	found org.slf4j#slf4j-api;1.7.36 in central
	found com.microsoft.azure#azure-keyvault-core;1.0.0 in central
	found com.google.guava#guava;27.0-jre in central
	found com.google.guava#failureaccess;1.0 in central
	found com.google.guava#listenablefuture;9999.0-empty-to-avoid-conflict-with-guava in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found org.checkerframework#checker-qual;2.5.2 in central
	found com.google.errorpron

Preconnectors ran successfully and connected to 'clientdatastorage' storage accounts
 'policy_df, claim_df, member_df, provider_df' are ready to use from container 'raw data'(meridian architecture of bronze) 


In [ ]:
import sys
import importlib

# Add the Silver layer directory to sys.path BEFORE importing
silver_dir = "/Users/manojrammopati/Public/Projects/bupa_insurance_project/_02_Silver"
if silver_dir not in sys.path:
    sys.path.insert(0, silver_dir)

# Now import (and reload to pick up edits)
import utils_silver
from utils_silver import *

importlib.reload(utils_silver)

paths_bronze, paths_silver, paths_gold = utils_silver.build_paths(storage_account1)
print(sorted(paths_silver.keys()))
print("Imported utils_silver from", utils_silver.__file__)

✅ utils_silver.py loaded
✅ utils_silver.py loaded
['_metrics', '_quarantine', '_ref_dim_channel', '_ref_dim_product_line', '_reference', 'claims', 'members', 'policies', 'providers']
Imported utils_silver from /Users/manojrammopati/Public/Projects/bupa_insurance_project/02_Silver_Layer/utils_silver.py


# Cell 1 – Imports & config

In [3]:
from pyspark.sql import functions as F

# Databases
DB_GOLD   = "bupa_gold"
DB_SILVER = "bupa_silver"  # in case we use providers_silver

# Storage (Gold container)
GOLD_BASE = "abfss://golddata@clientdatastorage.dfs.core.windows.net"

DM_CLAIMS_EXP_PATH = f"{GOLD_BASE}/dm_claims_experience"

print("DB_GOLD:", DB_GOLD)
print("GOLD_BASE:", GOLD_BASE)
print("DM_CLAIMS_EXP_PATH:", DM_CLAIMS_EXP_PATH)


DB_GOLD: bupa_gold
GOLD_BASE: abfss://golddata@clientdatastorage.dfs.core.windows.net
DM_CLAIMS_EXP_PATH: abfss://golddata@clientdatastorage.dfs.core.windows.net/dm_claims_experience


# Cell 2 – Load fact_claims (+ optional providers)

In [4]:
# Replace with your actual storage account and path
fact_policies = spark.read.format("delta").load("abfss://golddata@clientdatastorage.dfs.core.windows.net/fact_policies")
fact_members  = spark.read.format("delta").load("abfss://golddata@clientdatastorage.dfs.core.windows.net/fact_members")
fact_claims   = spark.read.format("delta").load("abfss://golddata@clientdatastorage.dfs.core.windows.net/fact_claims")

In [5]:
DB_GOLD = "bupa_gold"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_GOLD}")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB_GOLD}.fact_policies
    USING DELTA
    LOCATION 'abfss://golddata@clientdatastorage.dfs.core.windows.net/fact_policies'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB_GOLD}.fact_members
    USING DELTA
    LOCATION 'abfss://golddata@clientdatastorage.dfs.core.windows.net/fact_members'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB_GOLD}.fact_claims
    USING DELTA
    LOCATION 'abfss://golddata@clientdatastorage.dfs.core.windows.net/fact_claims'
""")

DataFrame[]

In [6]:
# Core source: Gold fact_claims
claims = spark.table(f"{DB_GOLD}.fact_claims")

print("fact_claims rowcount:", claims.count())
claims.show(5, truncate=False)
claims.printSchema()


25/12/06 20:45:49 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


fact_claims rowcount: 558211


+---------+-----------+----------+-------------+------------+----------+------------------+-----------+----------+------------+-------------------+--------------+----------------------+--------------------+--------------+-------------+
|Claim_ID |Provider_ID|Member_Key|Date_Reported|Date_Settled|Payout_GBP|Claim_Amount_GBP  |Fraud_Label|Claim_Type|Claim_Status|Provider_Fraud_Flag|Days_To_Settle|Payout_to_Amount_Ratio|High_Cost_Claim_Flag|dq_money_valid|dq_date_valid|
+---------+-----------+----------+-------------+------------+----------+------------------+-----------+----------+------------+-------------------+--------------+----------------------+--------------------+--------------+-------------+
|CLM110013|PRV51790   |BENE15435 |NULL         |NULL        |200.0     |244.07511199775644|0          |Travel    |Settled     |0                  |NULL          |0.8194198841618822    |0                   |1             |1            |
|CLM110020|PRV53679   |BENE53030 |NULL         |NULL    

# (Optional provider enrichment – only if table exists; safe try/except)

In [8]:
df_providers = spark.read.format("delta").load("abfss://silverdata@clientdatastorage.dfs.core.windows.net/providers")

In [9]:
DB_GOLD = "bupa_silver"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_GOLD}")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB_GOLD}.providers
    USING DELTA
    LOCATION 'abfss://silverdata@clientdatastorage.dfs.core.windows.net/providers'
""")

DataFrame[]

In [10]:
try:
    providers = spark.table(f"{DB_SILVER}.providers")
    HAS_PROVIDERS = True
    print("✅ Loaded providers from", f"{DB_SILVER}.providers")
except Exception as e:
    providers = None
    HAS_PROVIDERS = False
    print("⚠️ No providers table found, skipping provider enrichment. Details:", e)


✅ Loaded providers from bupa_silver.providers


# Cell 3 – Derived claim-level experience fields

Here we keep one row per claim (Claim_ID) but add rich features:

severity / cost bands

high-cost flag

settlement SLA buckets

open vs closed

payout ratio & exposure

In [11]:
# Threshold for "high cost" claims (you can tune this)
HIGH_COST_THRESHOLD = 5000.0

claims_enriched = (
    claims
    # Money exposure: difference between billed vs paid
    .withColumn(
        "Exposure_GBP",
        F.when(
            F.col("Claim_Amount_GBP").isNotNull() & F.col("Payout_GBP").isNotNull(),
            F.col("Claim_Amount_GBP") - F.col("Payout_GBP")
        ).otherwise(F.lit(None).cast("double"))
    )
    # Payout ratio (how much we actually paid vs claimed)
    .withColumn(
        "Payout_Ratio",
        F.when(
            F.col("Claim_Amount_GBP").isNotNull() & (F.col("Claim_Amount_GBP") > 0),
            F.col("Payout_GBP") / F.col("Claim_Amount_GBP")
        ).otherwise(F.lit(None).cast("double"))
    )
    # High cost flag
    .withColumn(
        "High_Cost_Flag",
        F.when(F.col("Claim_Amount_GBP") >= HIGH_COST_THRESHOLD, 1).otherwise(0)
    )
    # Cost band
    .withColumn(
        "Cost_Band",
        F.when(F.col("Claim_Amount_GBP").isNull(), "Unknown")
         .when(F.col("Claim_Amount_GBP") < 500, "<500")
         .when(F.col("Claim_Amount_GBP") < 2000, "500–2k")
         .when(F.col("Claim_Amount_GBP") < 5000, "2k–5k")
         .when(F.col("Claim_Amount_GBP") < 10000, "5k–10k")
         .otherwise("10k+")
    )
    # SLA band based on Days_To_Settle
    .withColumn(
        "Settlement_SLA_Band",
        F.when(F.col("Days_To_Settle").isNull(), "Unknown")
         .when(F.col("Days_To_Settle") <= 7,  "≤ 1 week")
         .when(F.col("Days_To_Settle") <= 30, "1–4 weeks")
         .when(F.col("Days_To_Settle") <= 90, "1–3 months")
         .otherwise("> 3 months")
    )
    # Open vs closed at a glance
    .withColumn(
        "Open_Closed_Flag",
        F.when(F.col("Claim_Status").isin("Pending"), "Open")
         .otherwise("Closed")
    )
)

claims_enriched.select(
    "Claim_ID", "Claim_Type", "Claim_Status",
    "Claim_Amount_GBP", "Payout_GBP", "Exposure_GBP",
    "High_Cost_Flag", "Cost_Band", "Settlement_SLA_Band",
    "Open_Closed_Flag", "Fraud_Label"
).show(10, truncate=False)

claims_enriched.printSchema()


+---------+----------+------------+------------------+----------+------------------+--------------+---------+-------------------+----------------+-----------+
|Claim_ID |Claim_Type|Claim_Status|Claim_Amount_GBP  |Payout_GBP|Exposure_GBP      |High_Cost_Flag|Cost_Band|Settlement_SLA_Band|Open_Closed_Flag|Fraud_Label|
+---------+----------+------------+------------------+----------+------------------+--------------+---------+-------------------+----------------+-----------+
|CLM110013|Travel    |Settled     |244.07511199775644|200.0     |44.07511199775644 |0             |<500     |Unknown            |Closed          |0          |
|CLM110020|Hospital  |Settled     |4117.6845027538575|2600.0    |1517.6845027538575|0             |2k–5k    |Unknown            |Closed          |0          |
|CLM110025|Dental    |Settled     |105.71412122348184|80.0      |25.71412122348184 |0             |<500     |Unknown            |Closed          |0          |
|CLM110026|NULL      |Settled     |4690.566401

# Cell 4 – Optional provider enrichment (fraud / risk)

If providers are available, we bring in provider fraud flag so BI can slice claims by provider risk.

In [18]:
# Read a sample of data from the bupa_silver.providers table
providers=spark.table("bupa_silver.providers")
providers.show(10, truncate=False)

+-----------+----------+------------------+
|Provider_ID|Fraud_Flag|Fraud_Label_Source|
+-----------+----------+------------------+
|PRV54400   |0         |labelled          |
|PRV55039   |1         |labelled          |
|PRV53872   |0         |labelled          |
|PRV53223   |0         |labelled          |
|PRV54888   |0         |labelled          |
|PRV54182   |0         |labelled          |
|PRV55099   |0         |labelled          |
|PRV51262   |0         |labelled          |
|PRV51665   |0         |labelled          |
|PRV55541   |0         |labelled          |
+-----------+----------+------------------+
only showing top 10 rows



In [20]:
claims_enriched.show(5, truncate=False)

+---------+-----------+----------+-------------+------------+----------+------------------+-----------+----------+------------+-------------------+--------------+----------------------+--------------------+--------------+-------------+------------------+------------------+--------------+---------+-------------------+----------------+
|Claim_ID |Provider_ID|Member_Key|Date_Reported|Date_Settled|Payout_GBP|Claim_Amount_GBP  |Fraud_Label|Claim_Type|Claim_Status|Provider_Fraud_Flag|Days_To_Settle|Payout_to_Amount_Ratio|High_Cost_Claim_Flag|dq_money_valid|dq_date_valid|Exposure_GBP      |Payout_Ratio      |High_Cost_Flag|Cost_Band|Settlement_SLA_Band|Open_Closed_Flag|
+---------+-----------+----------+-------------+------------+----------+------------------+-----------+----------+------------+-------------------+--------------+----------------------+--------------------+--------------+-------------+------------------+------------------+--------------+---------+-------------------+----------

In [21]:
from pyspark.sql import functions as F

# Check if fact_claims already has Provider_Fraud_Flag
HAS_PROVIDER_FLAG_IN_FACT = "Provider_Fraud_Flag" in claims_enriched.columns

if HAS_PROVIDERS:
    providers_small = (
        providers
        .select(
            F.col("Provider_ID").alias("Prov_ID"),
            F.col("Fraud_Flag").alias("Prov_Provider_Fraud_Flag")
        )
        .dropDuplicates(["Prov_ID"])
    )

    # Join in a controlled way so we don't create duplicate column names
    joined = (
        claims_enriched.alias("c")
        .join(
            providers_small.alias("p"),
            F.col("c.Provider_ID") == F.col("p.Prov_ID"),
            how="left"
        )
    )

    if HAS_PROVIDER_FLAG_IN_FACT:
        # Keep the existing Provider_Fraud_Flag from fact_claims
        # and add the provider-table version as a separate column for comparison
        dm_claims_base = joined.select(
            F.col("c.*"),
            F.col("p.Prov_Provider_Fraud_Flag").alias("Provider_Fraud_Flag_Ref")
        )
    else:
        # No flag in fact_claims – use provider one as the main column
        dm_claims_base = joined.select(
            F.col("c.*"),
            F.col("p.Prov_Provider_Fraud_Flag").alias("Provider_Fraud_Flag")
        )

    print("Joined with providers; rowcount:", dm_claims_base.count())
else:
    dm_claims_base = claims_enriched
    print("Using claims_enriched without providers; rowcount:", dm_claims_base.count())


Joined with providers; rowcount: 558211


In [22]:
preview_cols = ["Claim_ID", "Provider_ID", "Fraud_Label", "High_Cost_Flag"]

if "Provider_Fraud_Flag" in dm_claims_base.columns:
    preview_cols.append("Provider_Fraud_Flag")

if "Provider_Fraud_Flag_Ref" in dm_claims_base.columns:
    preview_cols.append("Provider_Fraud_Flag_Ref")

dm_claims_base.select(*preview_cols).show(10, truncate=False)


+---------+-----------+-----------+--------------+-------------------+-----------------------+
|Claim_ID |Provider_ID|Fraud_Label|High_Cost_Flag|Provider_Fraud_Flag|Provider_Fraud_Flag_Ref|
+---------+-----------+-----------+--------------+-------------------+-----------------------+
|CLM110013|PRV51790   |0          |0             |0                  |0                      |
|CLM110020|PRV53679   |0          |0             |0                  |0                      |
|CLM110025|PRV53238   |0          |0             |0                  |0                      |
|CLM110026|PRV57345   |0          |0             |1                  |1                      |
|CLM110041|PRV52058   |0          |0             |0                  |0                      |
|CLM110063|PRV53884   |0          |0             |1                  |1                      |
|CLM110067|PRV51836   |0          |0             |1                  |1                      |
|CLM110074|PRV55525   |0          |0             |

# Cell 5 – Write dm_claims_experience (Delta + table)

In [23]:
# 1) Write to Gold container
(
    dm_claims_base
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(DM_CLAIMS_EXP_PATH)
)
print("✅ dm_claims_experience written to:", DM_CLAIMS_EXP_PATH)

# 2) Register as Gold table
spark.sql("CREATE DATABASE IF NOT EXISTS bupa_gold")
spark.sql("DROP TABLE IF EXISTS bupa_gold.dm_claims_experience")

spark.sql(f"""
CREATE TABLE bupa_gold.dm_claims_experience
USING DELTA
LOCATION '{DM_CLAIMS_EXP_PATH}'
""")

print("✅ Registered table: bupa_gold.dm_claims_experience")

# 3) Quick sanity check
spark.table("bupa_gold.dm_claims_experience").show(10, truncate=False)
spark.table("bupa_gold.dm_claims_experience").printSchema()


✅ dm_claims_experience written to: abfss://golddata@clientdatastorage.dfs.core.windows.net/dm_claims_experience
✅ Registered table: bupa_gold.dm_claims_experience


+---------+-----------+----------+-------------+------------+----------+------------------+-----------+----------+------------+-------------------+--------------+----------------------+--------------------+--------------+-------------+------------------+------------------+--------------+---------+-------------------+----------------+-----------------------+
|Claim_ID |Provider_ID|Member_Key|Date_Reported|Date_Settled|Payout_GBP|Claim_Amount_GBP  |Fraud_Label|Claim_Type|Claim_Status|Provider_Fraud_Flag|Days_To_Settle|Payout_to_Amount_Ratio|High_Cost_Claim_Flag|dq_money_valid|dq_date_valid|Exposure_GBP      |Payout_Ratio      |High_Cost_Flag|Cost_Band|Settlement_SLA_Band|Open_Closed_Flag|Provider_Fraud_Flag_Ref|
+---------+-----------+----------+-------------+------------+----------+------------------+-----------+----------+------------+-------------------+--------------+----------------------+--------------------+--------------+-------------+------------------+------------------+-------

# Cell A – Basic profile of dm_claims_experience

In [24]:
dm = spark.table("bupa_gold.dm_claims_experience")

print("Rowcount (dm_claims_experience):", dm.count())
print("Distinct Claim_ID:", dm.select("Claim_ID").distinct().count())
print("Distinct Provider_ID:", dm.select("Provider_ID").distinct().count())

dm.select(
    "Claim_ID", "Claim_Type", "Claim_Status",
    "Claim_Amount_GBP", "Payout_GBP", "Exposure_GBP",
    "High_Cost_Flag", "Cost_Band", "Settlement_SLA_Band",
    "Open_Closed_Flag", "Fraud_Label", "Provider_Fraud_Flag"
).show(10, truncate=False)


Rowcount (dm_claims_experience): 558211


Distinct Claim_ID: 558211


Distinct Provider_ID: 5394


+---------+----------+------------+------------------+----------+------------------+--------------+---------+-------------------+----------------+-----------+-------------------+
|Claim_ID |Claim_Type|Claim_Status|Claim_Amount_GBP  |Payout_GBP|Exposure_GBP      |High_Cost_Flag|Cost_Band|Settlement_SLA_Band|Open_Closed_Flag|Fraud_Label|Provider_Fraud_Flag|
+---------+----------+------------+------------------+----------+------------------+--------------+---------+-------------------+----------------+-----------+-------------------+
|CLM110013|Travel    |Settled     |244.07511199775644|200.0     |44.07511199775644 |0             |<500     |Unknown            |Closed          |0          |0                  |
|CLM110020|Hospital  |Settled     |4117.6845027538575|2600.0    |1517.6845027538575|0             |2k–5k    |Unknown            |Closed          |0          |0                  |
|CLM110025|Dental    |Settled     |105.71412122348184|80.0      |25.71412122348184 |0             |<500  

# Cell B – Experience by Claim_Type & Fraud_Label

This replicates and extends the style of the summary you saw earlier in fact_claims.

In [25]:
print("▶ Claim experience by Claim_Type")

(
    dm.groupBy("Claim_Type")
      .agg(
          F.count("*").alias("n_claims"),
          F.avg("Payout_GBP").alias("avg_payout"),
          F.avg("Claim_Amount_GBP").alias("avg_claim_amount"),
          F.sum("High_Cost_Flag").alias("n_high_cost_claims")
      )
      .orderBy(F.col("n_claims").desc())
      .show(truncate=False)
)

print("▶ Claim experience by Fraud_Label")

(
    dm.groupBy("Fraud_Label")
      .agg(
          F.count("*").alias("n_claims"),
          F.avg("Payout_GBP").alias("avg_payout"),
          F.avg("Claim_Amount_GBP").alias("avg_claim_amount"),
          F.sum("High_Cost_Flag").alias("n_high_cost_claims")
      )
      .orderBy("Fraud_Label")
      .show(truncate=False)
)


▶ Claim experience by Claim_Type


+----------+--------+------------------+------------------+------------------+
|Claim_Type|n_claims|avg_payout        |avg_claim_amount  |n_high_cost_claims|
+----------+--------+------------------+------------------+------------------+
|Hospital  |234059  |1008.2132710128643|1309.4962742599596|13890             |
|Outpatient|185088  |997.5023772475795 |1295.7349725932131|10956             |
|Dental    |71612   |965.2528905769983 |1254.32477252121  |4178              |
|NULL      |39196   |991.5185222981937 |1291.4997691162691|2316              |
|Maternity |21604   |992.9346417330124 |1289.8630929445965|1282              |
|Travel    |5280    |960.6628787878788 |1253.4169020673085|296               |
|Physio    |1372    |1038.7172011661808|1314.7430930603496|77                |
+----------+--------+------------------+------------------+------------------+

▶ Claim experience by Fraud_Label


+-----------+--------+-----------------+------------------+------------------+
|Fraud_Label|n_claims|avg_payout       |avg_claim_amount  |n_high_cost_claims|
+-----------+--------+-----------------+------------------+------------------+
|0          |549656  |996.7822783704717|1295.028775841741 |32449             |
|1          |8555    |1011.780245470485|1313.6701153614936|546               |
+-----------+--------+-----------------+------------------+------------------+



# Cell C – SLA & cost pressure view

In [26]:
print("▶ Settlement SLA distribution")

(
    dm.groupBy("Settlement_SLA_Band")
      .agg(
          F.count("*").alias("n_claims"),
          F.avg("Days_To_Settle").alias("avg_days_to_settle")
      )
      .orderBy("n_claims", ascending=False)
      .show(truncate=False)
)

print("▶ Cost_Band vs Open/Closed")

(
    dm.groupBy("Cost_Band", "Open_Closed_Flag")
      .agg(
          F.count("*").alias("n_claims"),
          F.sum("High_Cost_Flag").alias("n_high_cost_claims")
      )
      .orderBy("Cost_Band", "Open_Closed_Flag")
      .show(truncate=False)
)


▶ Settlement SLA distribution


+-------------------+--------+------------------+
|Settlement_SLA_Band|n_claims|avg_days_to_settle|
+-------------------+--------+------------------+
|Unknown            |558211  |NULL              |
+-------------------+--------+------------------+

▶ Cost_Band vs Open/Closed


+---------+----------------+--------+------------------+
|Cost_Band|Open_Closed_Flag|n_claims|n_high_cost_claims|
+---------+----------------+--------+------------------+
|10k+     |Closed          |13633   |13633             |
|10k+     |Open            |4921    |4921              |
|2k–5k    |Closed          |22776   |0                 |
|2k–5k    |Open            |8111    |0                 |
|500–2k   |Closed          |48251   |0                 |
|500–2k   |Open            |17264   |0                 |
|5k–10k   |Closed          |10636   |10636             |
|5k–10k   |Open            |3805    |3805              |
|<500     |Closed          |315718  |0                 |
|<500     |Open            |113096  |0                 |
+---------+----------------+--------+------------------+



# Cell D – Dashboard KPI checklist for Claims Experience

This is just a printed list you can directly talk through in interviews or implement in Power BI/Tableau.

In [27]:
kpis = [
    "1) Claims volume & average payout by Claim_Type",
    "2) High-cost claims count and % by Claim_Type and Fraud_Label",
    "3) Average Payout_Ratio by Claim_Type (how much we actually pay vs billed)",
    "4) Distribution of Settlement_SLA_Band (how fast we settle claims)",
    "5) Exposure_GBP totals by Claim_Status (how much is not paid vs billed)",
    "6) Open vs Closed claims by Cost_Band (pipeline of big tickets)",
    "7) If providers available: claims & payout by Provider_Fraud_Flag",
    "8) High_Cost_Flag trend over time (if we add a claim month later)",
]

print("📊 Recommended Claims Experience KPIs based on dm_claims_experience:")
for k in kpis:
    print("  -", k)

print("""
This mart keeps a single row per claim, enriched with:
  • cost bands and high-cost flags
  • settlement SLA buckets
  • payout ratio and exposure
  • open vs closed status
  • (optionally) provider fraud risk

It is designed for BI dashboards and for feeding ML feature stores
(e.g. fraud models, severity prediction, operational workload management).
""")


📊 Recommended Claims Experience KPIs based on dm_claims_experience:
  - 1) Claims volume & average payout by Claim_Type
  - 2) High-cost claims count and % by Claim_Type and Fraud_Label
  - 3) Average Payout_Ratio by Claim_Type (how much we actually pay vs billed)
  - 4) Distribution of Settlement_SLA_Band (how fast we settle claims)
  - 5) Exposure_GBP totals by Claim_Status (how much is not paid vs billed)
  - 6) Open vs Closed claims by Cost_Band (pipeline of big tickets)
  - 7) If providers available: claims & payout by Provider_Fraud_Flag
  - 8) High_Cost_Flag trend over time (if we add a claim month later)

This mart keeps a single row per claim, enriched with:
  • cost bands and high-cost flags
  • settlement SLA buckets
  • payout ratio and exposure
  • open vs closed status
  • (optionally) provider fraud risk

It is designed for BI dashboards and for feeding ML feature stores
(e.g. fraud models, severity prediction, operational workload management).

